# Notebook 02: Registry Analysis (RQ1)

Generates all RQ1 evidence:
- Table 1: Signal registry summary statistics
- Table 2: ISO 704 criterion outcomes per confirmed signal
- Appendix A: Full registry export CSV
- Warrant pass-rate analysis

**No ML models required.** Runs in under 30 seconds.

Run order: this notebook can be run independently of all others.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

# Ensure project root is on path
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tpc.registry.loader import load_registry, registry_summary
from tpc.registry.iso704_validator import assess_canonical, assess_tortured_incoherence
from tpc.registry.warrant import assess_signal_file

SIGNALS_DIR = ROOT / 'signals' / 'phrases'
APPENDIX_DIR = ROOT / 'data' / 'appendix'
FIGURES_DIR = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

print('Setup complete. Registry path:', SIGNALS_DIR)

## 1. Load registry and compute summary statistics (Table 1)

In [ ]:
# Load all statuses for full picture
all_signals = load_registry(
    status_filter=('confirmed', 'candidate', 'deprecated'),
    signals_dir=SIGNALS_DIR
)
confirmed_signals = [s for s in all_signals if s.status.value == 'confirmed']

summary = registry_summary(all_signals)

print('=== Table 1: Signal Registry Summary ===')
rows = [
    ('Total signals (all statuses)', summary['total']),
    ('Confirmed', summary['by_status'].get('confirmed', 0)),
    ('Candidate', summary['by_status'].get('candidate', 0)),
    ('Deprecated', summary['by_status'].get('deprecated', 0)),
    ('Domains represented', ', '.join(summary['by_domain'].keys())),
    ('Mean precision on clean corpus (confirmed)', f"{summary['mean_precision']:.4f}"),
    ('Mean recall on retracted corpus (confirmed)', f"{summary['mean_recall']:.4f}"),
    ('Total corpus sightings', summary['total_sightings']),
    ('Literary warrant pass rate', f"{summary['warrant_pass_rates']['literary']:.1%}"),
    ('Terminological warrant pass rate', f"{summary['warrant_pass_rates']['terminological']:.1%}"),
    ('Statistical warrant pass rate', f"{summary['warrant_pass_rates']['statistical']:.1%}"),
]
df_table1 = pd.DataFrame(rows, columns=['Metric', 'Value'])
print(df_table1.to_string(index=False))

# Save for paper
df_table1.to_csv(APPENDIX_DIR / 'table1_registry_summary.csv', index=False)
print('\nSaved: data/appendix/table1_registry_summary.csv')

## 2. Per-signal warrant assessment (for Table 1 detail)

In [ ]:
# Detailed per-signal warrant breakdown
warrant_rows = []
for yaml_path in sorted(SIGNALS_DIR.rglob('*.yaml')):
    assessment = assess_signal_file(yaml_path)
    raw = yaml.safe_load(yaml_path.read_text())
    warrant_rows.append({
        'signal_id':              raw['id'],
        'tortured':               raw['tortured'],
        'canonical':              raw['canonical'],
        'domain':                 raw['domain'],
        'status':                 raw['status'],
        'literary_satisfied':     assessment.literary.result.value,
        'literary_sightings':     assessment.literary.independent_sightings,
        'terminological_satisfied': assessment.terminological.result.value,
        'domain_expert_orcid':    assessment.terminological.domain_expert_orcid,
        'statistical_satisfied':  assessment.statistical.result.value,
        'precision_on_clean':     assessment.statistical.precision_on_clean,
        'recall_on_retracted':    assessment.statistical.recall_on_retracted,
        'recommended_status':     assessment.recommended_status,
    })

df_warrants = pd.DataFrame(warrant_rows)
print('=== Per-signal Warrant Assessment ===')
print(df_warrants[['signal_id','tortured','literary_satisfied',
                   'terminological_satisfied','statistical_satisfied',
                   'precision_on_clean','recall_on_retracted']].to_string(index=False))

## 3. ISO 704 criterion outcomes per signal (Table 2)

In [ ]:
iso_rows = []
for yaml_path in sorted(SIGNALS_DIR.rglob('*.yaml')):
    raw = yaml.safe_load(yaml_path.read_text())
    if raw.get('status') != 'confirmed':
        continue
    result = assess_canonical(raw['canonical'], raw['domain'])
    
    # Cross-check with stored expert assessment
    stored = raw.get('warrant', {}).get('terminological', {}).get('iso704_criteria', {})
    
    iso_rows.append({
        'Signal ID':      raw['id'],
        'Tortured':       raw['tortured'],
        'Canonical':      raw['canonical'],
        'Precision':      '✓' if stored.get('precision') else '✗',
        'Economy':        '✓' if stored.get('economy') else '✗',
        'Appropriateness':'✓' if stored.get('appropriateness') else '✗',
        'Consistency':    '✓' if stored.get('consistency') else '✗',
        'Transparency':   '✓' if stored.get('transparency') else '✗*' if not result.transparency else '✓',
        'Auto-screen':    '✓' if result.is_valid_canonical else '⚠',
        'Expert valid':   '✓',  # All confirmed signals have passed expert review
    })

df_iso = pd.DataFrame(iso_rows)
print('=== Table 2: ISO 704 Criterion Outcomes ===')
print(df_iso.to_string(index=False))

df_iso.to_csv(APPENDIX_DIR / 'table2_iso704_assessment.csv', index=False)
print('\nSaved: data/appendix/table2_iso704_assessment.csv')

## 4. Tortured phrase incoherence analysis

In [ ]:
# For each confirmed signal, show how severely the tortured form
# violates ISO 704 criteria relative to the canonical
incoherence_rows = []
for yaml_path in sorted(SIGNALS_DIR.rglob('*.yaml')):
    raw = yaml.safe_load(yaml_path.read_text())
    if raw.get('status') != 'confirmed':
        continue
    result = assess_tortured_incoherence(
        tortured=raw['tortured'],
        canonical=raw['canonical'],
        domain=raw['domain']
    )
    incoherence_rows.append({
        'id':         raw['id'],
        'tortured':   raw['tortured'],
        'canonical':  raw['canonical'],
        'violations': '; '.join(result['violations']) if result['violations'] else 'none',
        'severity':   result['severity'],
    })

df_incoherence = pd.DataFrame(incoherence_rows)
print('=== Tortured Phrase Incoherence Assessment ===')
print(df_incoherence.to_string(index=False))

## 5. Export Appendix A (full registry CSV)

In [ ]:
appendix_rows = []
for yaml_path in sorted(SIGNALS_DIR.rglob('*.yaml')):
    raw = yaml.safe_load(yaml_path.read_text())
    w = raw.get('warrant', {})
    appendix_rows.append({
        'signal_id':                raw['id'],
        'tortured':                 raw['tortured'],
        'canonical':                raw['canonical'],
        'domain':                   raw['domain'],
        'status':                   raw['status'],
        'warrant_literary':         w.get('literary', {}).get('satisfied', False),
        'warrant_terminological':   w.get('terminological', {}).get('satisfied', False),
        'warrant_statistical':      w.get('statistical', {}).get('satisfied', False),
        'precision_on_clean':       w.get('statistical', {}).get('precision_on_clean'),
        'recall_on_retracted':      w.get('statistical', {}).get('recall_on_retracted'),
        'retracted_papers':         raw.get('prevalence', {}).get('retracted_papers', 0),
        'legitimate_papers':        raw.get('prevalence', {}).get('legitimate_papers', 0),
        'discovery_date':           raw.get('discovery_date'),
        'paraphrase_tool_origin':   raw.get('paraphrase_tool_origin'),
    })

df_appendix_a = pd.DataFrame(appendix_rows)
df_appendix_a.to_csv(APPENDIX_DIR / 'appendix_a_registry_export.csv', index=False)
print('Appendix A saved:', len(df_appendix_a), 'signals')
print(df_appendix_a)

## 6. Registry composition visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# (a) signals by domain
domain_counts = df_appendix_a[df_appendix_a['status'] == 'confirmed']['domain'].value_counts()
axes[0].barh(domain_counts.index, domain_counts.values, color='#2166ac')
axes[0].set_xlabel('Confirmed signals')
axes[0].set_title('(a) Signals by domain')

# (b) precision distribution
prec = df_appendix_a['precision_on_clean'].dropna()
axes[1].hist(prec, bins=10, color='#2166ac', edgecolor='white', range=(0.9, 1.0))
axes[1].axvline(x=0.95, color='#d73027', linestyle='--', label='Gate threshold (0.95)')
axes[1].set_xlabel('Precision on clean corpus')
axes[1].set_title('(b) Precision distribution')
axes[1].legend(fontsize=8)

# (c) warrant type pass rates
warrant_rates = [
    summary['warrant_pass_rates']['literary'],
    summary['warrant_pass_rates']['terminological'],
    summary['warrant_pass_rates']['statistical'],
]
warrant_labels = ['Literary\nwarrant', 'Terminological\nwarrant', 'Statistical\nwarrant']
bars = axes[2].bar(warrant_labels, [r * 100 for r in warrant_rates], color=['#4393c3','#2166ac','#053061'])
axes[2].set_ylabel('Pass rate (%)')
axes[2].set_ylim(0, 110)
axes[2].set_title('(c) Warrant gate pass rates')
for bar, rate in zip(bars, warrant_rates):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{rate:.0%}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Figure: Registry composition and warrant quality (RQ1)', y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_rq1_registry_quality.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved: figures/fig_rq1_registry_quality.pdf')